# RDD Fundamentals

## Module 2 · Notebook 1

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define** what an RDD (Resilient Distributed Dataset) is and explain its role as Spark's foundational abstraction
2. **Describe** the five key properties that define every RDD
3. **Create** RDDs using `parallelize()`, `textFile()`, and transformations on existing RDDs
4. **Compare** RDDs vs DataFrames vs Datasets and articulate when to use each
5. **Work** with the RDD type system including PairRDDs
6. **Inspect** RDD metadata: partitions, lineage, and data distribution

---

## Why Study RDDs?

Even though most modern Spark applications use DataFrames and the Structured API, **every DataFrame operation eventually compiles down to RDD operations**. Understanding RDDs gives you:

- A mental model for **how Spark distributes and processes data**
- The ability to **debug performance issues** by reading execution plans
- Knowledge of **fault tolerance** mechanisms (lineage-based recovery)
- Skills for cases where the **DataFrame API is insufficient** (custom partitioning, low-level control)

Think of RDDs as the assembly language of Spark — you don't write in it every day, but knowing it makes you a far better engineer.

---

## 1⃣ What is an RDD?

**RDD** stands for **Resilient Distributed Dataset**. Let's unpack each word:

| Word | Meaning |
|------|--------|
| **Resilient** | Fault-tolerant — if a partition is lost, Spark can recompute it from its lineage (the chain of transformations that created it) |
| **Distributed** | Data is split into **partitions** spread across multiple machines in the cluster |
| **Dataset** | A collection of records (elements) — could be integers, strings, tuples, or any Python object |

```
┌─────────────────────────────────────────────────────────┐
│                        RDD                              │
│                                                         │
│   ┌──────────┐  ┌──────────┐  ┌──────────┐             │
│   │Partition 0│  │Partition 1│  │Partition 2│  ...       │
│   │[a, b, c] │  │[d, e, f] │  │[g, h, i] │             │
│   └──────────┘  └──────────┘  └──────────┘             │
│       Worker 1      Worker 1      Worker 2              │
│                                                         │
│   • Immutable (read-only)                               │
│   • Lazily evaluated                                    │
│   • Type-safe in Scala; dynamically typed in Python     │
└─────────────────────────────────────────────────────────┘
```

### Key Characteristics

1. **Immutable**: Once created, an RDD cannot be modified. Transformations create new RDDs.
2. **Lazily Evaluated**: Transformations (like `map`, `filter`) are not executed immediately — they build up a computation plan (DAG). Only **actions** (like `collect`, `count`) trigger execution.
3. **Partitioned**: Data is divided into chunks (partitions) that can be processed in parallel across the cluster.
4. **Typed**: Each RDD holds elements of a specific type (e.g., `RDD[str]`, `RDD[Tuple[str, int]]`).

---

## 2⃣ The Five Properties of an RDD

Every RDD is defined by exactly **five properties**. These are the building blocks Spark uses internally:

```
┌─────────────────────────────────────────────────────────────────┐
│                    RDD: Five Properties                         │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. partitions()      → List of partitions (chunks of data)     │
│                                                                 │
│  2. dependencies()    → List of parent RDDs this RDD depends on │
│                                                                 │
│  3. compute(partition) → Function to compute data for a         │
│                          given partition                        │
│                                                                 │
│  4. partitioner        → (Optional) How keys are partitioned    │
│                          e.g., HashPartitioner, RangePartitioner │
│                                                                 │
│  5. preferredLocations(partition)                                │
│                        → (Optional) Where to place computation  │
│                          for data locality                      │
└─────────────────────────────────────────────────────────────────┘
```

| # | Property | Purpose | Example |
|---|----------|---------|--------|
| 1 | **Partitions** | Define the units of parallelism | An RDD with 4 partitions can process 4 chunks in parallel |
| 2 | **Dependencies** | Link to parent RDDs (lineage) | `filteredRDD` depends on `originalRDD` |
| 3 | **Compute** | The logic to produce data | `map(lambda x: x*2)` defines the compute function |
| 4 | **Partitioner** | Key distribution strategy | `HashPartitioner(10)` distributes keys across 10 partitions |
| 5 | **Preferred Locations** | Data locality hints |"Process partition 0 on node that has HDFS block 0"|

Let's see these properties in action once we create our first RDD.

---

## Setup: Create SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("02_RDD_01 - RDD Fundamentals")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# Get a reference to the SparkContext — the gateway to RDD operations
sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"SparkContext master: {sc.master}")
print(f"Default parallelism: {sc.defaultParallelism}")

> **Note**: `sc = spark.sparkContext` gives us the `SparkContext`, which is the original Spark entry point and the gateway to all RDD operations. In Spark 1.x, you created a `SparkContext` directly. In modern Spark, you create a `SparkSession`, which internally creates the `SparkContext` for you.

---

## 3⃣ Creating RDDs

There are **three ways** to create an RDD:

```
                    Creating RDDs
                    ─────────────
                         │
           ┌─────────────┼─────────────┐
           │             │             │
     ┌─────┴─────┐ ┌────┴────┐ ┌──────┴──────┐
     │parallelize│ │textFile │ │Transforming │
     │ (from     │ │ (from   │ │ (from       │
     │  memory)  │ │  file)  │ │  other RDD) │
     └───────────┘ └─────────┘ └─────────────┘
```

### 3.1 `sc.parallelize()` — From an In-Memory Collection

In [ ]:
# Create an RDD from a Python list
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
numbers_rdd = sc.parallelize(numbers)

print(f"Type: {type(numbers_rdd)}")
print(f"RDD: {numbers_rdd}")
# Notice: no data is printed! The RDD is lazily evaluated.
# The RDD only knows HOW to compute the data, not WHAT the data is.

In [ ]:
# To see the data, we need an ACTION like collect()
result = numbers_rdd.collect()
print(f"Data: {result}")
print(f"Number of partitions: {numbers_rdd.getNumPartitions()}")

In [ ]:
# You can specify the number of partitions
rdd_2parts = sc.parallelize(numbers, numSlices=2)
rdd_5parts = sc.parallelize(numbers, numSlices=5)

print(f"2 partitions: {rdd_2parts.getNumPartitions()}")
print(f"5 partitions: {rdd_5parts.getNumPartitions()}")

# glom() collects data per-partition, so we can see distribution
print(f"\n2-partition distribution: {rdd_2parts.glom().collect()}")
print(f"5-partition distribution: {rdd_5parts.glom().collect()}")

> **`glom()`** is a powerful inspection tool. It returns an RDD where each element is a list containing all elements of a partition. We'll use it throughout these notebooks to understand data distribution.

In [ ]:
# parallelize works with any Python objects
strings_rdd = sc.parallelize(["hello", "world", "spark", "is", "awesome"])
tuples_rdd = sc.parallelize([("Alice", 30), ("Bob", 25), ("Charlie", 35)])
mixed_rdd = sc.parallelize([1, "two", 3.0, True, None])  # Not recommended!

print(f"Strings: {strings_rdd.collect()}")
print(f"Tuples: {tuples_rdd.collect()}")
print(f"Mixed: {mixed_rdd.collect()}")

### 3.2 `sc.textFile()` — From External Files

In [ ]:
# Read a text file — each line becomes an element
words_rdd = sc.textFile("/opt/spark-data/words.txt")

print(f"Type: {type(words_rdd)}")
print(f"Number of partitions: {words_rdd.getNumPartitions()}")
print(f"\nFirst 5 lines:")
for line in words_rdd.take(5):
    print(f"  '{line}'")

In [ ]:
# Read a CSV file as raw text (each line is a string)
sales_rdd = sc.textFile("/opt/spark-data/sales.csv")

print(f"Number of lines: {sales_rdd.count()}")
print(f"\nFirst 3 lines:")
for line in sales_rdd.take(3):
    print(f"  '{line}'")

In [ ]:
# You can specify minimum partitions for textFile
words_4parts = sc.textFile("/opt/spark-data/words.txt", minPartitions=4)
print(f"Partitions (minPartitions=4): {words_4parts.getNumPartitions()}")

# Note: Spark may create more partitions than requested, never fewer

### 3.3 `sc.wholeTextFiles()` — Read Files as Key-Value Pairs

In [ ]:
# wholeTextFiles reads each file as a single record: (filename, content)
files_rdd = sc.wholeTextFiles("/opt/spark-data/words.txt")

# Each element is (filepath, entire_file_content)
for filepath, content in files_rdd.collect():
    print(f"File: {filepath}")
    print(f"Content length: {len(content)} chars")
    print(f"First 100 chars: {content[:100]}...")

### 3.4 From Transforming Existing RDDs

In [ ]:
# Create a new RDD by transforming an existing one
base_rdd = sc.parallelize([1, 2, 3, 4, 5])

# Each transformation creates a NEW RDD (RDDs are immutable)
doubled_rdd = base_rdd.map(lambda x: x * 2)        # [2, 4, 6, 8, 10]
filtered_rdd = base_rdd.filter(lambda x: x > 3)    # [4, 5]
squared_rdd = doubled_rdd.map(lambda x: x ** 2)     # [4, 16, 36, 64, 100]

print(f"Base:     {base_rdd.collect()}")
print(f"Doubled:  {doubled_rdd.collect()}")
print(f"Filtered: {filtered_rdd.collect()}")
print(f"Squared:  {squared_rdd.collect()}")

# base_rdd was not modified!
print(f"\nBase (unchanged): {base_rdd.collect()}")

### 3.5 From DataFrames (`.rdd` property)

In [ ]:
# You can also get an RDD from a DataFrame
df = spark.read.csv("/opt/spark-data/sales.csv", header=True, inferSchema=True)
df.show(5)

# Convert DataFrame to RDD — each row becomes a Row object
df_rdd = df.rdd
print(f"\nType: {type(df_rdd)}")
print(f"First element: {df_rdd.first()}")
print(f"Element type: {type(df_rdd.first())}")

---

## 4⃣ Inspecting the Five Properties

Now let's inspect each of the five properties on a real RDD.

In [ ]:
# Create an RDD we'll inspect
data = list(range(1, 101))  # 1 to 100
inspect_rdd = sc.parallelize(data, numSlices=4)

# Property 1: Partitions
print("═" * 60)
print("PROPERTY 1: PARTITIONS")
print("═" * 60)
print(f"Number of partitions: {inspect_rdd.getNumPartitions()}")
print(f"Partition sizes: {[len(p) for p in inspect_rdd.glom().collect()]}")
print(f"\nPartition 0 contents: {inspect_rdd.glom().collect()[0][:10]}...")

In [ ]:
# Property 2: Dependencies (lineage)
print("═" * 60)
print("PROPERTY 2: DEPENDENCIES")
print("═" * 60)

# Build a chain of transformations
step1 = inspect_rdd.filter(lambda x: x % 2 == 0)   # even numbers
step2 = step1.map(lambda x: x * 10)                  # multiply by 10
step3 = step2.filter(lambda x: x > 200)              # keep only > 200

# toDebugString() shows the entire lineage
print("Lineage of step3:")
print(step3.toDebugString().decode('utf-8'))

In [ ]:
# Property 3: Compute function (implicit - we define it via transformations)
print("═" * 60)
print("PROPERTY 3: COMPUTE FUNCTION")
print("═" * 60)
print("The compute function is the transformation logic itself.")
print("For step3, the compute chain is:")
print("  parallelize → filter(even) → map(×10) → filter(>200)")
print(f"\nResult: {step3.collect()[:10]}...")

In [ ]:
# Property 4: Partitioner
print("═" * 60)
print("PROPERTY 4: PARTITIONER")
print("═" * 60)

# Regular RDDs have no partitioner
print(f"inspect_rdd partitioner: {inspect_rdd.partitioner}")

# PairRDDs can have partitioners after certain operations
pair_rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("b", 4)])
print(f"pair_rdd partitioner: {pair_rdd.partitioner}")

# After partitioning operations like partitionBy, a partitioner is set
partitioned_rdd = pair_rdd.partitionBy(2)
print(f"partitioned_rdd partitioner: {partitioned_rdd.partitioner}")

In [ ]:
# Property 5: Preferred Locations
print("═" * 60)
print("PROPERTY 5: PREFERRED LOCATIONS")
print("═" * 60)
print("Preferred locations are data locality hints.")
print("For in-memory RDDs (parallelize), there are usually no preferences.")
print("For HDFS files, Spark prefers to process data on the node")
print("that already has the data block — minimizing network transfer.")
print("\nThis is the concept of 'moving computation to data'")
print("rather than 'moving data to computation'.")

---

## 5⃣ RDD vs DataFrame vs Dataset

Spark has three main data abstractions. Understanding the differences is crucial:

```
Evolution of Spark APIs:

  Spark 1.0 (2014)    Spark 1.3 (2015)    Spark 1.6 (2016)    Spark 2.0+ (2016+)
  ┌──────────┐        ┌──────────┐        ┌──────────┐        ┌──────────────────┐
  │   RDD    │───────►│DataFrame │───────►│ Dataset  │───────►│DataFrame=Dataset │
  │          │        │(SchemaRDD│        │          │        │   (unified API)  │
  │Low-level │        │ with SQL)│        │Type-safe │        │                  │
  └──────────┘        └──────────┘        └──────────┘        └──────────────────┘
```

| Feature | RDD | DataFrame | Dataset (Scala/Java) |
|---------|-----|-----------|---------------------|
| **Abstraction Level** | Low-level | High-level | High-level + type-safe |
| **Schema** | No schema | Has schema (columns, types) | Schema + compile-time type checking |
| **Optimization** | No Catalyst/Tungsten | Full Catalyst + Tungsten | Full Catalyst + Tungsten |
| **Serialization** | Java serialization (slow) | Tungsten binary format (fast) | Tungsten + Encoders |
| **API Style** | Functional (map, filter) | Declarative (select, where) | Both |
| **Python Support** |  Full |  Full |  Not available |
| **Best For** | Custom logic, unstructured data | Structured data, SQL | Scala type-safe operations |

### Performance Comparison

```
Performance (higher is better):

DataFrame ██████████████████████████████  Best (Catalyst + Tungsten)
Dataset   ████████████████████████████   Close to DataFrame
RDD       ████████████████               Slower (no optimization)
```

### When to Use RDDs

 **Use RDDs when:**
- You need fine-grained control over physical data distribution
- Your data is unstructured (not rows/columns)
- You need custom partitioning logic
- You're doing complex aggregations not expressible in SQL
- You're working with legacy Spark code

 **Don't use RDDs when:**
- You can express the operation in DataFrame/SQL (most cases)
- Performance is critical (DataFrames are usually faster)
- You're doing standard ETL (read → transform → write)

In [ ]:
# Practical comparison: same operation, RDD vs DataFrame

# --- RDD approach ---
sales_rdd = sc.textFile("/opt/spark-data/sales.csv")
header = sales_rdd.first()
rdd_result = (
    sales_rdd
    .filter(lambda line: line != header)           # skip header
    .map(lambda line: line.split(","))             # split into fields
    .map(lambda fields: (fields[1], float(fields[3]) * float(fields[4])))  # (region, revenue)
    .reduceByKey(lambda a, b: a + b)               # sum by region
    .collect()
)
print("RDD result (revenue by region):")
for region, revenue in sorted(rdd_result):
    print(f"  {region}: ${revenue:,.2f}")

In [ ]:
# --- DataFrame approach (same result, much cleaner) ---
from pyspark.sql import functions as F

df = spark.read.csv("/opt/spark-data/sales.csv", header=True, inferSchema=True)
df_result = (
    df
    .withColumn("revenue", F.col("units") * F.col("unit_price"))
    .groupBy("region")
    .agg(F.sum("revenue").alias("total_revenue"))
    .orderBy("region")
)
print("DataFrame result (revenue by region):")
df_result.show()

---

## 6⃣ The RDD Type System

In PySpark, RDDs can hold any Python object. However, certain operations require **Pair RDDs** — RDDs whose elements are 2-tuples `(key, value)`.

```
Regular RDD:    [1, 2, 3, 4, 5]
                ["hello","world"]

Pair RDD:       [("a", 1), ("b", 2), ("a", 3)]
                 ─key─  ─val─

Operations only available on Pair RDDs:
  • reduceByKey   • groupByKey    • sortByKey
  • join          • cogroup       • aggregateByKey
  • keys()        • values()      • mapValues()
  • partitionBy()
```

In [ ]:
# Creating Pair RDDs

# Method 1: parallelize with tuples
pair_rdd = sc.parallelize([("apple", 3), ("banana", 2), ("apple", 5), ("banana", 1)])
print(f"Pair RDD: {pair_rdd.collect()}")

# Method 2: map a regular RDD to create pairs
words = sc.parallelize(["hello", "world", "hello", "spark", "world", "hello"])
word_pairs = words.map(lambda w: (w, 1))
print(f"Word pairs: {word_pairs.collect()}")

In [ ]:
# Pair RDD operations

# keys() and values()
print(f"Keys: {pair_rdd.keys().collect()}")
print(f"Values: {pair_rdd.values().collect()}")

# reduceByKey — aggregate values by key
totals = pair_rdd.reduceByKey(lambda a, b: a + b)
print(f"\nSum by key: {totals.collect()}")

# groupByKey — group values by key
grouped = pair_rdd.groupByKey()
print(f"\nGrouped by key:")
for key, values in grouped.collect():
    print(f"  {key}: {list(values)}")

In [ ]:
# mapValues — transform only the values, keeping keys unchanged
doubled_values = pair_rdd.mapValues(lambda v: v * 2)
print(f"Doubled values: {doubled_values.collect()}")

# sortByKey
sorted_rdd = pair_rdd.sortByKey()
print(f"Sorted by key: {sorted_rdd.collect()}")

# countByKey — action that returns a dict
counts = pair_rdd.countByKey()
print(f"Count by key: {dict(counts)}")

---

## 7⃣ Word Count: The Classic RDD Example

Word Count is the"Hello World"of distributed computing. Let's build it step by step to see how RDD operations compose:

```
Input File          flatMap           map              reduceByKey
──────────         ──────────       ──────────        ──────────
"hello world"→"hello"→  ("hello", 1)  →  ("hello", 2)
"hello spark"→"world"→  ("world", 1)  →  ("world", 1)
"hello"→  ("hello", 1)  →  ("spark", 1)
"spark"→  ("spark", 1)
```

In [ ]:
# Step-by-step Word Count

# Step 1: Read the file
lines = sc.textFile("/opt/spark-data/words.txt")
print("Step 1 - Lines:")
for line in lines.take(5):
    print(f"  '{line}'")

In [ ]:
# Step 2: Split each line into words (flatMap: one-to-many)
words = lines.flatMap(lambda line: line.lower().split())
print(f"Step 2 - First 10 words: {words.take(10)}")

In [ ]:
# Step 3: Map each word to (word, 1)
word_ones = words.map(lambda word: (word, 1))
print(f"Step 3 - First 5 pairs: {word_ones.take(5)}")

In [ ]:
# Step 4: Reduce by key to count
word_counts = word_ones.reduceByKey(lambda a, b: a + b)
print("Step 4 - Word counts:")
for word, count in word_counts.sortBy(lambda x: -x[1]).collect():
    print(f"  {word}: {count}")

In [ ]:
# The complete pipeline in one chain
result = (
    sc.textFile("/opt/spark-data/words.txt")
    .flatMap(lambda line: line.lower().split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: -x[1])
    .collect()
)

print("Top 10 words:")
for word, count in result[:10]:
    print(f"  {word:15s} {count}")

---

## 8⃣ Internal Investigation: Understanding What's Happening

Let's use Spark's built-in tools to peek under the hood.

In [ ]:
# Inspecting RDD lineage with toDebugString()
# This is the RDD equivalent of DataFrame's explain()

pipeline = (
    sc.textFile("/opt/spark-data/words.txt")
    .flatMap(lambda line: line.lower().split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)

print("═" * 60)
print("RDD LINEAGE (toDebugString)")
print("═" * 60)
print(pipeline.toDebugString().decode('utf-8'))

**Reading the lineage output:**

```
(8) PythonRDD[...] at ...              ← Final RDD (8 partitions)
 |  MapPartitionsRDD[...] at ...       ← After shuffle (reduceByKey)
 |  ShuffledRDD[...] at ...            ← Shuffle boundary
 +-(2) PairwiseRDD[...] at ...         ← Before shuffle (2 partitions)
    |  PythonRDD[...] at ...           ← map/flatMap
    |  /opt/spark-data/words.txt       ← Source file
```

- Numbers in parentheses `(8)` indicate partition count
- `|` means narrow dependency (no shuffle)
- `+-` means the start of a new stage (shuffle boundary)

In [ ]:
# Inspecting partition contents with glom()
small_rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8], numSlices=3)

print("Data distribution across partitions:")
partitions = small_rdd.glom().collect()
for i, partition in enumerate(partitions):
    bar = "█" * len(partition)
    print(f"  Partition {i}: {partition:20s}  {bar} ({len(partition)} elements)")

In [ ]:
# mapPartitionsWithIndex — see which partition processes which data
def show_partition(index, iterator):
    """Yield (partition_index, element) for each element."""
    for element in iterator:
        yield (index, element)

data_rdd = sc.parallelize(range(20), numSlices=4)
partition_info = data_rdd.mapPartitionsWithIndex(show_partition).collect()

print("Element → Partition mapping:")
for part_idx, element in partition_info:
    print(f"  Element {element:2d} → Partition {part_idx}")

In [ ]:
# RDD ID and name
named_rdd = sc.parallelize([1, 2, 3])
named_rdd.setName("my_numbers")

print(f"RDD id: {named_rdd.id()}")
print(f"RDD name: {named_rdd.name()}")
print(f"Is cached: {named_rdd.is_cached}")
print(f"Storage level: {named_rdd.getStorageLevel()}")

---

## 9⃣ Experiment: Partition Count and Data Distribution

**Hypothesis**: When we `parallelize()` a list, Spark distributes elements as evenly as possible across partitions.

Let's test this with different partition counts.

In [ ]:
data = list(range(1, 21))  # 20 elements

for num_parts in [1, 2, 3, 4, 5, 7, 10, 20]:
    rdd = sc.parallelize(data, numSlices=num_parts)
    partitions = rdd.glom().collect()
    sizes = [len(p) for p in partitions]

    # Visual representation
    bars = " ".join([f"P{i}:{'█' * s}({s})" for i, s in enumerate(sizes)])
    print(f"\n{num_parts:2d} partitions: {bars}")

**Observation**: Spark distributes elements as evenly as possible. When 20 elements don't divide evenly by the partition count, some partitions get one extra element.

This is important because:
- **Uneven partitions = wasted resources** (some tasks finish earlier and sit idle)
- **Too few partitions = underutilization** (not enough parallelism)
- **Too many partitions = overhead** (scheduling cost per task)

---

## Experiment: RDD vs DataFrame Performance

Let's measure the performance difference between RDD and DataFrame operations on the same dataset.

In [ ]:
import time

# Generate a larger dataset for meaningful timing
large_data = [(f"key_{i % 100}", i) for i in range(100000)]

# --- RDD approach ---
rdd = sc.parallelize(large_data, 8)

start = time.time()
rdd_result = rdd.reduceByKey(lambda a, b: a + b).collect()
rdd_time = time.time() - start
print(f"RDD reduceByKey: {rdd_time:.3f}s, {len(rdd_result)} keys")

# --- DataFrame approach ---
df = spark.createDataFrame(large_data, ["key", "value"])

start = time.time()
df_result = df.groupBy("key").sum("value").collect()
df_time = time.time() - start
print(f"DataFrame groupBy: {df_time:.3f}s, {len(df_result)} keys")

print(f"\nDataFrame is {rdd_time/df_time:.1f}x faster" if df_time < rdd_time else f"\nRDD is {df_time/rdd_time:.1f}x faster")

> **Note**: In production with larger data, DataFrames are typically **2-10x faster** than RDDs because:
> 1. **Catalyst optimizer** generates an efficient execution plan
> 2. **Tungsten** uses off-heap memory and efficient binary formats
> 3. **Whole-stage code generation** compiles operations into optimized JVM bytecode
> 4. **No Python serialization overhead** for DataFrame operations

---

## Exercises

### Exercise 1: Basic RDD Creation (Beginner)

Create an RDD containing the numbers 1 through 50, with exactly 5 partitions. Then:
1. Print the number of partitions
2. Use `glom()` to show how many elements are in each partition
3. Collect and print the first 10 elements

In [ ]:
# YOUR CODE HERE


### Exercise 2: Text File Processing (Beginner)

Read `/opt/spark-data/words.txt` and:
1. Count the total number of lines
2. Count the total number of words (split each line on whitespace)
3. Find the longest line (by character count)

In [ ]:
# YOUR CODE HERE


### Exercise 3: Pair RDD from CSV (Intermediate)

Read `/opt/spark-data/sales.csv` as a text file and create a Pair RDD of `(product, total_units)`. Then:
1. Find the product with the most total units sold
2. Count how many distinct regions are in the data
3. Display results sorted by total units descending

In [ ]:
# YOUR CODE HERE


### Exercise 4: Lineage Investigation (Intermediate)

Build a chain of at least 4 transformations on an RDD:
1. Start with numbers 1-100
2. Apply filter, map, flatMap, and reduceByKey in any order
3. Print the lineage using `toDebugString()`
4. Identify which transformations cause stage boundaries (shuffles)

In [ ]:
# YOUR CODE HERE


### Exercise 5: RDD vs DataFrame Challenge (Advanced)

Using `/opt/spark-data/retail/orders.csv`:
1. Implement a"top 5 customers by total order amount"query using **pure RDD operations** (textFile, map, filter, reduceByKey, etc.)
2. Implement the same query using DataFrames
3. Compare the code complexity and discuss which approach is better

In [ ]:
# YOUR CODE HERE


---

## Solutions

<details>
<summary>Click to expand solutions</summary>

### Solution 1

```python
rdd = sc.parallelize(range(1, 51), numSlices=5)
print(f"Partitions: {rdd.getNumPartitions()}")
print(f"Elements per partition: {[len(p) for p in rdd.glom().collect()]}")
print(f"First 10: {rdd.take(10)}")
```

### Solution 2

```python
lines = sc.textFile("/opt/spark-data/words.txt")
print(f"Lines: {lines.count()}")
print(f"Words: {lines.flatMap(lambda l: l.split()).count()}")
print(f"Longest line: {lines.reduce(lambda a, b: a if len(a) > len(b) else b)}")
```

### Solution 3

```python
sales = sc.textFile("/opt/spark-data/sales.csv")
header = sales.first()
data = sales.filter(lambda line: line != header)

# (product, units)
product_units = data.map(lambda l: l.split(",")).map(lambda f: (f[2], int(f[3])))
totals = product_units.reduceByKey(lambda a, b: a + b)

# Most units
top = totals.sortBy(lambda x: -x[1]).first()
print(f"Top product: {top}")

# Distinct regions
regions = data.map(lambda l: l.split(",")[1]).distinct().count()
print(f"Distinct regions: {regions}")

# Sorted results
for prod, units in totals.sortBy(lambda x: -x[1]).collect():
    print(f"{prod}: {units}")
```

### Solution 4

```python
pipeline = (
    sc.parallelize(range(1, 101))
    .filter(lambda x: x % 3 == 0)             # narrow
    .map(lambda x: (x % 10, x))               # narrow
    .flatMap(lambda kv: [(kv[0], kv[1]), (kv[0] + 10, kv[1] * 2)])  # narrow
    .reduceByKey(lambda a, b: a + b)           # wide (shuffle!)
)
print(pipeline.toDebugString().decode('utf-8'))
# The reduceByKey causes a stage boundary (ShuffledRDD)
```

### Solution 5

```python
# RDD approach
orders_rdd = sc.textFile("/opt/spark-data/retail/orders.csv")
header = orders_rdd.first()
top5_rdd = (
    orders_rdd
    .filter(lambda l: l != header)
    .map(lambda l: l.split(","))
    .map(lambda f: (f[1], float(f[3])))  # (customer_id, amount)
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: -x[1])
    .take(5)
)
print("RDD Top 5:", top5_rdd)

# DataFrame approach
orders_df = spark.read.csv("/opt/spark-data/retail/orders.csv", header=True, inferSchema=True)
top5_df = (
    orders_df
    .groupBy("customer_id")
    .agg(F.sum("amount").alias("total"))
    .orderBy(F.desc("total"))
    .limit(5)
)
top5_df.show()
```

</details>

---

## Summary

### Key Takeaways

| Concept | Key Point |
|---------|----------|
| **RDD** | Resilient Distributed Dataset — Spark's fundamental data abstraction |
| **Immutability** | RDDs cannot be modified; transformations create new RDDs |
| **Lazy Evaluation** | Transformations build a plan; actions execute it |
| **Five Properties** | Partitions, dependencies, compute, partitioner, preferred locations |
| **Pair RDDs** | RDDs of (key, value) tuples enabling group-by and join operations |
| **Lineage** | Chain of dependencies that enables fault tolerance |

### Important Terms

- **SparkContext (sc)**: The entry point for RDD operations
- **Partition**: A chunk of data processed by a single task
- **Transformation**: A lazy operation that creates a new RDD (map, filter, etc.)
- **Action**: An eager operation that triggers computation (collect, count, etc.)
- **DAG**: Directed Acyclic Graph — the computation plan
- **Lineage**: The chain of RDD transformations from source to result

### Common Mistakes

1.  Calling `collect()` on large RDDs → brings all data to the driver, causing OOM
2.  Using RDDs when DataFrames would work → missing out on Catalyst optimization
3.  Forgetting that RDDs are lazy → wondering why your `map()` doesn't print anything
4.  Not setting partition count → getting too many or too few partitions

### Interview Questions

1. **What does RDD stand for? Explain each word.**
2. **What are the five properties of an RDD?**
3. **How does Spark achieve fault tolerance with RDDs?** (Through lineage — it can recompute lost partitions)
4. **What is the difference between a transformation and an action?**
5. **When would you use RDDs instead of DataFrames?**
6. **What is a Pair RDD and why is it important?**

### What's Next?

In the next notebook, we'll do a **deep dive into RDD Transformations and Actions** — covering every major operation with examples and performance analysis.

In [ ]:
# Clean up
spark.stop()
print("SparkSession stopped.")